# Backpropagation from derivatives to autograd

Can a hand-derived neural-network gradient agree with PyTorch and finite differences?

I derive a two-layer classifier, check its gradients in double precision and train it with an explicit update loop. The image task makes tensor shapes visible while keeping the derivative checks small.

## Run settings

The default run uses CPU and a small training budget. Set `PROJECT_FULL=1` before opening the notebook for the longer run. Both modes use the same data split and evaluation code. A short run checks the experiment; the number of epochs limits what its scores can establish.

In [3]:
import json
import os
from pathlib import Path
import sys

import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F

root = Path.cwd()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "pyproject.toml").exists():
    raise FileNotFoundError("Open this notebook inside the repository")
sys.path.insert(0, str(root / "src"))
from deep_projects.training import (
    classification_scores,
    digits_data,
    predict,
    seed_everything,
)

QUICK = os.environ.get("PROJECT_FULL", "0") != "1"
EPOCHS = 6 if QUICK else 40
seed_everything(1031)
output = root / "figures" / "01_autograd_and_backprop"
output.mkdir(parents=True, exist_ok=True)
(root / "results").mkdir(exist_ok=True)
print({"torch": torch.__version__, "device": "cpu", "epochs": EPOCHS, "seed": 1031})

{'torch': '2.14.1+cu130', 'device': 'cpu', 'epochs': 40, 'seed': 1031}


## Matrix convention

Rows represent images. The network is $H=\tanh(XW_1+b_1)$ and $Z=HW_2+b_2$. I use tanh so finite differences avoid ReLU's nondifferentiable zero. Cross-entropy is averaged over observations, so its logit gradient is $(P-Y)/n$.

In [5]:
data = digits_data(8)
train_images, train_labels = data.train.tensors
features = train_images[:5].flatten(1).double()
labels = train_labels[:5]
generator = torch.Generator().manual_seed(1031)
first_weight = (
    torch.randn(64, 12, generator=generator, dtype=torch.float64) * 0.05
).requires_grad_()
first_bias = torch.zeros(12, dtype=torch.float64, requires_grad=True)
second_weight = (
    torch.randn(12, 10, generator=generator, dtype=torch.float64) * 0.05
).requires_grad_()
second_bias = torch.zeros(10, dtype=torch.float64, requires_grad=True)
hidden = torch.tanh(features @ first_weight + first_bias)
logits = hidden @ second_weight + second_bias
loss = F.cross_entropy(logits, labels)
parameters = [first_weight, first_bias, second_weight, second_bias]
automatic = torch.autograd.grad(loss, parameters)
print(features.shape, hidden.shape, logits.shape, float(loss.detach()))

torch.Size([5, 64]) torch.Size([5, 12]) torch.Size([5, 10]) 2.303888059905602


## Manual chain rule

The hidden derivative multiplies the backpropagated gradient by one minus tanh squared. Bias gradients sum over images. I compare every parameter entry, not only its norm, because a transposition can hide in summary statistics.

In [7]:
with torch.no_grad():
    logit_gradient = (logits.softmax(-1) - F.one_hot(labels, 10)) / len(labels)
    second_gradient = hidden.T @ logit_gradient
    hidden_gradient = (logit_gradient @ second_weight.T) * (1 - hidden.square())
    manual = [
        features.T @ hidden_gradient,
        hidden_gradient.sum(0),
        second_gradient,
        logit_gradient.sum(0),
    ]
errors = []
for name, expected, actual in zip(["W1", "b1", "W2", "b2"], manual, automatic):
    error = float((expected - actual).abs().max())
    errors.append({"parameter": name, "max_absolute_error": error})
    assert torch.allclose(expected, actual, atol=1e-10, rtol=1e-8)
print(pd.DataFrame(errors))

  parameter  max_absolute_error
0        W1        1.040834e-17
1        b1        1.387779e-17
2        W2        6.938894e-18
3        b2        5.551115e-17


## Central finite difference

For one weight, the numerical derivative is $(\ell(w+h)-\ell(w-h))/(2h)$. Double precision and a moderate step reduce cancellation. This independent check catches an error shared by a hand-written backward implementation and its own tests.

In [9]:
def objective(weight):
    values = torch.tanh(features @ weight + first_bias.detach())
    return F.cross_entropy(
        values @ second_weight.detach() + second_bias.detach(), labels
    )


step = 1e-5
plus, minus = first_weight.detach().clone(), first_weight.detach().clone()
plus[3, 2] += step
minus[3, 2] -= step
numerical = float((objective(plus) - objective(minus)) / (2 * step))
analytical = float(automatic[0][3, 2])
assert abs(numerical - analytical) < 1e-8
print({"finite_difference": numerical, "autograd": analytical})

{'finite_difference': 0.005823891258138757, 'autograd': 0.005823891251653857}


## Explicit minibatch updates

The experiment below uses a PyTorch network but performs SGD updates directly under `no_grad`. Gradients are cleared before each minibatch. Validation selects a state dictionary; final testing happens once after restoring it.

In [11]:
import copy

model = nn.Sequential(nn.Flatten(), nn.Linear(64, 64), nn.Tanh(), nn.Linear(64, 10))
train_loader, validation_loader, test_loader = data.loaders()
rows, best_loss, best_weights = [], float("inf"), None
for epoch in range(EPOCHS):
    model.train()
    for images, labels in train_loader:
        model.zero_grad(set_to_none=True)
        loss = F.cross_entropy(model(images), labels)
        loss.backward()
        with torch.no_grad():
            for parameter in model.parameters():
                parameter.add_(parameter.grad, alpha=-0.15)
    val_logits, val_labels = predict(model, validation_loader)
    val_loss = float(F.cross_entropy(val_logits, val_labels))
    rows.append({"epoch": epoch + 1, "validation_loss": val_loss})
    if val_loss < best_loss:
        best_loss, best_weights = val_loss, copy.deepcopy(model.state_dict())
model.load_state_dict(best_weights)
logits, targets = predict(model, test_loader)
print(pd.DataFrame(rows))
print(classification_scores(logits, targets))
results = {
    "gradient_checks": errors,
    "finite_difference_error": abs(numerical - analytical),
    "test": classification_scores(logits, targets),
}

    epoch  validation_loss
0       1         2.026804
1       2         1.673976
2       3         1.295816
3       4         0.992900
4       5         0.783343
5       6         0.644359
6       7         0.545756
7       8         0.476846
8       9         0.426629
9      10         0.388415
10     11         0.354413
11     12         0.336242
12     13         0.314035
13     14         0.294152
14     15         0.279822
15     16         0.263918
16     17         0.251996
17     18         0.243553
18     19         0.238234
19     20         0.228620
20     21         0.222387
21     22         0.214378
22     23         0.207711
23     24         0.203459
24     25         0.199578
25     26         0.191692
26     27         0.188230
27     28         0.184472
28     29         0.182782
29     30         0.178023
30     31         0.176458
31     32         0.174564
32     33         0.171665
33     34         0.166804
34     35         0.165332
35     36         0.163929
3

## Reading the result

Gradient agreement validates this network's derivative convention. The trained classifier is a separate end-to-end check that the updates reduce validation loss. Correct differentiation alone does not guarantee a suitable optimization schedule.

Recorded run: The manual gradients agreed with autograd to approximately $10^{-17}$ in the double-precision check. The separate central finite-difference error was about $6.5 \times 10^{-12}$. These checks validate the matrix convention before training.

In [13]:
result_path = root / "results" / "01_autograd_and_backprop.json"
result_path.write_text(json.dumps(results, indent=2, allow_nan=False) + "\n")
print(result_path.relative_to(root))

results/01_autograd_and_backprop.json


## Next experiment

Replace tanh with ReLU and choose finite-difference points whose preactivations stay away from zero. Add a NumPy-only forward and backward path using the same matrix convention.

## Sources

UniPd Machine Learning for Human Data: uploaded Lab_nn and Labs 1 to 7, plus the [course data folder](https://drive.google.com/drive/folders/14xwb6leBeD3QMZH6wnnIIOi63D1D8xmS). Lab_nn. [HSE deep learning course](https://github.com/mryab/dl-hse-ami) and its [wiki syllabus](https://wiki.cs.hse.ru/Глубинное_обучение_2022). week01_intro and week02_backpropagation.

The implementations here were written for this project. Course notebooks and their saved weights are reference material; they are not redistributed.